# Phase 2 — SFT Retrain + Baseline Comparison

Two runs, identical config. Only variable: invalid class distribution.

| Run | Dataset | WandB name |
|-----|---------|------------|
| 1 | `dataset_final_qwen.jsonl` (curated, balanced) | `curated-3ep` |
| 2 | `dataset_baseline_qwen.jsonl` (random invalids) | `baseline-3ep` |

Build baseline first: `python ml/build_baseline_dataset.py`

In [ ]:
import torch
import wandb
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    TrainingArguments, Trainer, BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B"

def load_tokenizer():
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return tok

def build_prompt(sample, tokenizer):
    kernel_ver = sample.get("kernel_version", "unknown")
    assembly   = sample.get("verifier_log", "")
    if sample.get("is_valid", False):
        text = f"Kernel: {kernel_ver} | Status: VALID\n### ASSEMBLY:\n{assembly}{tokenizer.eos_token}"
    else:
        err = sample.get("error_reason_clean", "Unknown error")
        text = f"Kernel: {kernel_ver} | Status: INVALID | Error: {err}\n### ASSEMBLY:\n{assembly}{tokenizer.eos_token}"
    return {"formatted_prompt": text}

def tokenize(sample, tokenizer):
    enc = tokenizer(
        sample["formatted_prompt"],
        max_length=768,
        truncation=True,
        padding="max_length",
    )
    enc["labels"] = [
        l if l != tokenizer.pad_token_id else -100
        for l in enc["input_ids"]
    ]
    return enc

def load_model():
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb,
        device_map="auto",
        attn_implementation="sdpa",
    )
    model.gradient_checkpointing_enable()
    lora = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    )
    model = get_peft_model(model, lora)
    model.print_trainable_parameters()
    return model

def make_training_args(output_dir, wandb_name):
    return TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=3,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        bf16=True,
        optim="paged_adamw_8bit",
        logging_steps=10,
        save_steps=200,
        eval_strategy="steps",
        eval_steps=200,
        load_best_model_at_end=True,
        report_to="wandb",
        run_name=wandb_name,
    )

print("[*] Shared setup loaded.")

## Run 1 — Curated model
Run this cell, wait for it to finish, then run Run 2.

In [ ]:
DATASET_PATH = "/home/stefano-u/fuzzing_lab/shared_corpus/dataset_final_qwen.jsonl"
OUTPUT_DIR   = "/home/stefano-u/fuzzing_ml_env/modello_ebpf_curated_3ep"
WANDB_NAME   = "curated-3ep"

wandb.init(project="ebpf-thesis", name=WANDB_NAME, reinit=True)

tokenizer = load_tokenizer()

dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
split   = dataset.train_test_split(test_size=0.1, seed=42)
train   = split["train"].map(lambda s: build_prompt(s, tokenizer)).map(lambda s: tokenize(s, tokenizer))
val     = split["test"].map(lambda s: build_prompt(s, tokenizer)).map(lambda s: tokenize(s, tokenizer))

print(f"[*] Train: {len(train)}  Val: {len(val)}")

model   = load_model()
trainer = Trainer(
    model=model,
    args=make_training_args(OUTPUT_DIR, WANDB_NAME),
    train_dataset=train,
    eval_dataset=val,
)

trainer.train()
trainer.save_model(f"{OUTPUT_DIR}/adattatore_ebpf_v1")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/adattatore_ebpf_v1")
wandb.finish()
print(f"[+] Curated model saved → {OUTPUT_DIR}/adattatore_ebpf_v1")

## Run 2 — Baseline model
Build baseline first if not done: `python ml/build_baseline_dataset.py`

In [ ]:
DATASET_PATH = "/home/stefano-u/fuzzing_lab/shared_corpus/dataset_baseline_qwen.jsonl"
OUTPUT_DIR   = "/home/stefano-u/fuzzing_ml_env/modello_ebpf_baseline_3ep"
WANDB_NAME   = "baseline-3ep"

wandb.init(project="ebpf-thesis", name=WANDB_NAME, reinit=True)

tokenizer = load_tokenizer()

dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
split   = dataset.train_test_split(test_size=0.1, seed=42)
train   = split["train"].map(lambda s: build_prompt(s, tokenizer)).map(lambda s: tokenize(s, tokenizer))
val     = split["test"].map(lambda s: build_prompt(s, tokenizer)).map(lambda s: tokenize(s, tokenizer))

print(f"[*] Train: {len(train)}  Val: {len(val)}")

model   = load_model()
trainer = Trainer(
    model=model,
    args=make_training_args(OUTPUT_DIR, WANDB_NAME),
    train_dataset=train,
    eval_dataset=val,
)

trainer.train()
trainer.save_model(f"{OUTPUT_DIR}/adattatore_ebpf_v1")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/adattatore_ebpf_v1")
wandb.finish()
print(f"[+] Baseline model saved → {OUTPUT_DIR}/adattatore_ebpf_v1")